## 🔍 AIC 2026 — Thử nghiệm Vector Search Trực Quan

Notebook này cho phép bạn gõ một câu query văn bản, mã hoá thành vector 512d qua **LoRA-CLIP** (tự động nạp `lora_weights.pt` nếu có) và truy vấn trực tiếp lên **Qdrant Vector Database**. Kết quả sẽ hiển thị dưới dạng bảng Markdown trực quan.

### 1. Khởi tạo & Mã hóa Query với LoRA-CLIP

In [ ]:
import numpy as np
from backend.embedding.clip_encoder import encode_text_raw
from backend.config import USE_REMOTE_VECTOR_DB, QDRANT_HOST, QDRANT_PORT, QDRANT_COLLECTION_NAME

# Nhập câu truy vấn tìm kiếm
query_text = "a photo of a tree"

# Mã hóa text sang vector 512d (LoRA-CLIP được tự động áp dụng nếu lora_weights.pt tồn tại)
query_vector = encode_text_raw(query_text)
print(f"Text Query: '{query_text}'")
print(f"Vector Dim: {query_vector.shape}, Norm: {np.linalg.norm(query_vector):.4f}")

### 2. Tìm kiếm similarity trên Qdrant Vector Database

In [ ]:
from qdrant_client import QdrantClient

client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

results = client.search(
    collection_name=QDRANT_COLLECTION_NAME,
    query_vector=query_vector.tolist(),
    limit=10
)

print(f"Tìm thấy {len(results)} kết quả phù hợp nhất:")
for idx, hit in enumerate(results, 1):
    payload = hit.payload
    print(f"#{idx:02d} | Score: {hit.score:.4f} | Video: {payload.get('video_id')} | Frame ID: {payload.get('frame_id')} | Time: {payload.get('pts_time')}s")

### 3. Xuất bảng Markdown kết quả với định dạng nộp bài AIC

In [ ]:
from IPython.display import display, Markdown

md_lines = [f"### Kết quả tìm kiếm cho query = `{query_text}`:\n"]
md_lines.append("| Rank | Score | Video ID | Frame ID | Timestamp (s) | Submission Line |")
md_lines.append("| :---: | :---: | :---: | :---: | :---: | :---: |")

for idx, hit in enumerate(results, 1):
    p = hit.payload
    vid = p.get("video_id", "N/A")
    fid = p.get("frame_id", 0)
    pts = p.get("pts_time", 0.0)
    sub_line = f"`{vid}, {fid}`"
    md_lines.append(f"| **#{idx:02d}** | `{hit.score:.4f}` | {vid} | {fid} | {pts:.1f}s | {sub_line} |")

display(Markdown("\n".join(md_lines)))